In [1]:
from loaders import load_image_sensor_response, load_spectral_data

cameras = {
    "Raspberry Pi HQ camera": (
        load_image_sensor_response("image_sensor_data/imx477.csv"), 
        load_spectral_data("filter_data/Hoya cm500 (raspberry pi HQ camera).csv")
    ),
    "Starlight Eye": (
        load_image_sensor_response("image_sensor_data/imx585.csv"), 
        load_spectral_data("filter_data/unknown (starlight eye).csv")
    ),
    "One Inch Eye": (
        load_image_sensor_response("image_sensor_data/imx283.csv"), 
        load_spectral_data("filter_data/unknown (one inch eye).csv")
    ),
    "Axiom Beta": (
        load_image_sensor_response("image_sensor_data/cmv1200.csv"), 
        load_spectral_data("filter_data/UQG.csv")
    ),
}

loaded spectral data named 'imx477' from 400.0nm to 700.0nm
loaded spectral data named 'Hoya cm500 (raspberry pi HQ camera)' from 200.0nm to 1200.0nm
loaded spectral data named 'imx585' from 400.0nm to 1000.0nm
loaded spectral data named 'unknown (starlight eye)' from 350.0nm to 1090.0nm
loaded spectral data named 'imx283' from 400.0nm to 690.0nm
loaded spectral data named 'unknown (one inch eye)' from 390.0nm to 1100.0nm
loaded spectral data named 'cmv1200' from 300.0nm to 1100.0nm
loaded spectral data named 'UQG' from 300.0nm to 1100.0nm


In [6]:
import scipy.optimize
import numpy as np
import colour

color_checker = colour.SDS_COLOURCHECKERS['BabelColor Average']
human = colour.MSDS_CMFS['CIE 1931 2 Degree Standard Observer']
shape = colour.SpectralShape(380, 730, 5)


def print_yaml_matrix(matrix):
    for row in matrix:
        print("   " + " ".join(f"{f:.05f}," for f in row))

def align(sd):
    from copy import deepcopy
    return deepcopy(sd).align(shape)


def multiply_multi_sds(multi_sds, scalar_sd):
    return colour.MultiSpectralDistributions([sd * scalar_sd.align(sd.shape) for sd in multi_sds.to_sds()], name=multi_sds.name)


def simulate_exposure(stimulus, sensitivity):
    if isinstance(sensitivity, colour.MultiSpectralDistributions):
        return np.array([simulate_exposure(stimulus, s) for s in sensitivity.to_sds()])
    elif isinstance(sensitivity, colour.SpectralDistribution):
        sensitivity, stimulus = align(sensitivity), align(stimulus)
        return np.trapz(stimulus.values * sensitivity.values, x=sensitivity.wavelengths)
    else:
        raise TypeError(f"{sensitivity} is of unsupported type")

def find_raw_transform(raw, xyz):
    def cost(matrix):
        matrix = np.reshape(matrix, (3, 3))
        def difference_with_matrix_applied(raw, xyz):
            transformed = np.dot(matrix, raw)
            delta_e = colour.difference.delta_E(colour.XYZ_to_Lab(xyz), colour.XYZ_to_Lab(transformed), method="CIE 2000")
            return delta_e
        color_difference_sum = np.mean([difference_with_matrix_applied(raw, xyz) ** 6 for raw, xyz in zip(raw, xyz)])
        return color_difference_sum
    result = scipy.optimize.minimize(cost, np.ones((9,)))
    return np.reshape(result.x, (3, 3))

for name, (sensor, filter) in cameras.items():
    print(name)
    sensor_with_filter = multiply_multi_sds(sensor, filter)
    for illuminant in ['A', 'D65']:
        print(f"Illuminant {illuminant}")
        illuminant_sd = align(colour.SDS_ILLUMINANTS[illuminant])

        N_human = simulate_exposure(illuminant_sd, human.to_sds()[1])
        N_sensor = simulate_exposure(illuminant_sd, sensor_with_filter.to_sds()[1])

        xyz_data = []
        raw_data = []
        for patch in color_checker.data.keys():
            
            xyz_data.append((1 / N_human) * simulate_exposure(align(color_checker[patch]) * align(illuminant_sd), human))
            raw_data.append((1 / N_sensor) * simulate_exposure(align(color_checker[patch]) * align(illuminant_sd), sensor_with_filter))

        corr_matrix = find_raw_transform(raw_data, xyz_data)

        differences = {}
        for patch, xyz, raw in zip(color_checker.data.keys(), xyz_data, raw_data):
                differences[patch] = colour.difference.delta_E(colour.XYZ_to_Lab(xyz), colour.XYZ_to_Lab(np.dot(corr_matrix, raw)), method="CIE 2000")

        differences_list = list(differences.values())
        differences_keys = list(differences.keys())
        print(
            f"mean color distance: {np.mean(differences_list):.2f} " + 
            f"(min: {np.min(differences_list):.2f} @ {differences_keys[np.argmin(differences_list)]}; " + 
            f"max: {np.max(differences_list):.2f} @ {differences_keys[np.argmax(differences_list)]}; " + 
            f"std: {np.std(differences_list):.2f})"
        )

        print(f"XYZ -> raw:")
        print_yaml_matrix(np.linalg.inv(corr_matrix))
        print()

    print("\n------------------------------------------------------------------------------------------------------------\n")

Raspberry Pi HQ camera
Illuminant A
mean color distance: 0.73 (min: 0.36 @ moderate red; max: 1.35 @ red; std: 0.26)
XYZ -> raw:
   0.45421, -0.04675, -0.01252,
   -0.56831, 1.49509, 0.41018,
   -0.09866, 0.22752, 0.91775,

Illuminant D65
mean color distance: 1.32 (min: 0.25 @ green; max: 2.36 @ cyan; std: 0.60)
XYZ -> raw:
   0.34146, -0.01944, -0.02277,
   -0.55145, 1.28211, 0.24036,
   -0.13300, 0.25075, 0.67590,


------------------------------------------------------------------------------------------------------------

Starlight Eye
Illuminant A
mean color distance: 1.02 (min: 0.09 @ dark skin; max: 2.00 @ red; std: 0.57)
XYZ -> raw:
   1.34399, -0.50520, -0.05611,
   -0.16806, 1.08934, 0.30976,
   0.01209, 0.05688, 0.64977,

Illuminant D65
mean color distance: 1.32 (min: 0.76 @ neutral 8 (.23 d); max: 2.05 @ red; std: 0.37)
XYZ -> raw:
   1.08269, -0.32575, -0.12401,
   -0.25054, 1.09735, 0.15163,
   -0.04182, 0.12990, 0.50706,


------------------------------------------------